# 🩺 TriageCore - Faz 2: Google Colab GPU Fine-Tuning (QLoRA & PEFT)

Bu notebook, **TriageCore** projesinin 2. Fazı için hazırlanmıştır. 
Google Colab üzerinde sahip olduğunuz GPU donanımını (**A100**, **L4** veya **T4**) otomatik olarak algılar; A100/L4 ortamında **bfloat16** ve **FlashAttention-2** optimizasyonlarını devreye alır, T4 ortamında ise **float16** korumalı 4-bit NF4 QLoRA pipeline'ı ile bellek aşımı (OOM) yaşamadan modeli eğitir.

---

### 1. Adım: GPU Donanımı ve Mimari Tespiti (A100 / L4 / T4)

In [ ]:
# PyTorch kutuphanesini ice aktar
import torch

# Isletim sistemi komutunu calistirarak GPU ozetini ekrana yazdir
!nvidia-smi

# GPU modelini programatik olarak sorgula
device_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
# Kullaniciya algilanan GPU'yu bildir
print(f"[*] Algilanan GPU Mimarisi: {device_name}")

# A100 veya L4 gibi modern Ampere/Ada Lovelace GPU'lar bfloat16'yi native donanimsal destekler
is_bfloat16_supported = torch.cuda.is_bf16_supported()
# Kriter sonucunu konsola bas
print(f"[*] Bfloat16 Donanim Destegi: {'AKTIF (A100/L4)' if is_bfloat16_supported else 'PASIF (T4 - float16 kullanilacak)'}")

### 2. Adım: Gerekli Açık Kaynak Kütüphanelerin Kurulumu

In [ ]:
# Transformers kutuphanesini kur
!pip install -q transformers>=4.40.0
# PEFT (LoRA adaptor yonetimi) kutuphanesini kur
!pip install -q peft>=0.10.0
# 4-bit kuantizasyon motoru bitsandbytes'i kur
!pip install -q bitsandbytes>=0.43.0
# Supervised fine-tuning kutuphanesi TRL'yi kur
!pip install -q trl>=0.8.6
# Bellek ve dagitik egitim yoneticisi accelerate'i kur
!pip install -q accelerate>=0.29.0
# HuggingFace veri seti aracini kur
!pip install -q datasets>=2.18.0

### 3. Adım: 4-Bit NormalFloat (NF4) Kuantizasyon Yapılandırması

In [ ]:
# BitsAndBytesConfig sinifini ice aktar
from transformers import BitsAndBytesConfig

# GPU tipine gore hesaplama hassasiyetini dinamik belirle (A100/L4 icin bfloat16, T4 icin float16)
compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

# BitsAndBytes yapilandirma nesnesini olustur
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,                     # Modeli 4-bit agirliklarla VRAM'e yerlestir
    bnb_4bit_quant_type="nf4",             # Bilgi kaybi en dusuk olan NormalFloat4 tipini sec
    bnb_4bit_use_double_quant=True,         # Kuantizasyon sabitlerini de kuantize ederek ek bellek kazan
    bnb_4bit_compute_dtype=compute_dtype,  # Matris carpimlarinda dinamik secilen float tipini kullan
)

# Durumu ekrana yazdir
print(f"[*] BitsAndBytes 4-bit NF4 hazirlandi (Compute Dtype: {compute_dtype})")

### 4. Adım: Base Model ve Tokenizer'ın Yüklenmesi

In [ ]:
# Transformers bilesenlerini ice aktar
# PEFT k-bit egitim on hazirlik fonksiyonunu ice aktar
from peft import prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer

# Hedef acik agirlikli model adi
model_id = "meta-llama/Meta-Llama-3-8B-Instruct"

# Tokenizer'i hizli Rust motoruyla yukle
tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True, trust_remote_code=True)

# Pad token tanimli degilse eos token ile esitle
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
# Sagdan dolgu ayarini zorunlu kil
tokenizer.padding_side = "right"

# Base modeli 4-bit kuantize sekilde VRAM'e indir
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

# Modeli k-bit gradyan egitimine hazirla (LayerNorm stabilizasyonu)
model = prepare_model_for_kbit_training(model)
# Aktivasyon bellek tuketimini %60 azaltan gradient checkpointing'i ac
model.gradient_checkpointing_enable()

print("[*] Model basariyla VRAM'e yuklendi!")

### 5. Adım: All-Linear LoRA Katmanlarının Entegrasyonu

In [ ]:
# LoRA konfigurasyon ve adaptasyon fonksiyonlarini ice aktar
from peft import LoraConfig, get_peft_model

# LoRA hiperparametrelerini tanimla
lora_config = LoraConfig(
    r=16,                                    # Adaptor matris boyutu
    lora_alpha=32,                           # Olcekleme katsayisi
    target_modules=[                         # All-linear katmanlar
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_dropout=0.05,                       # %5 dropout
    bias="none",
    task_type="CAUSAL_LM",
)

# Modele LoRA adaptasyon katmanlarini monte et
model = get_peft_model(model, lora_config)
# Egitilebilir hafif parametre oranini konsola bas
model.print_trainable_parameters()

### 6. Adım: Veri Seti Yükleme ve SFTTrainer ile Eğitimi Başlatma

In [ ]:
# HuggingFace datasets kutuphanesinden veri yukleyiciyi ice aktar
from datasets import load_dataset

# Transformers egitim argumanlari sinifini ice aktar
from transformers import TrainingArguments

# TRL SFTTrainer sinifini ice aktar
from trl import SFTTrainer

# Veri dosyalarinin yollarini belirle
data_files = {"train": "train.jsonl", "validation": "val.jsonl"}
# JSONL dosyalarini yukle
dataset = load_dataset("json", data_files=data_files)

# A100 veya L4 varsa batch size'i artirabiliriz; T4 icin 2 idealdir
micro_batch = 4 if torch.cuda.is_bf16_supported() else 2
grad_accum = 4 if torch.cuda.is_bf16_supported() else 8

# Egitim parametrelerini belirle
training_args = TrainingArguments(
    output_dir="./triagecore-checkpoints",
    num_train_epochs=3,
    per_device_train_batch_size=micro_batch,
    gradient_accumulation_steps=grad_accum,
    optim="paged_adamw_8bit",
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    logging_steps=10,
    save_strategy="steps",
    save_steps=50,
    eval_strategy="steps",
    eval_steps=50,
    bf16=torch.cuda.is_bf16_supported(),      # A100/L4 icin bfloat16 aktif
    fp16=not torch.cuda.is_bf16_supported(),  # T4 icin float16 aktif
    max_grad_norm=0.3,
    report_to="none",
)

# SFTTrainer nesnesini ilklendir
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    peft_config=lora_config,
    max_seq_length=1024,
    tokenizer=tokenizer,
    args=training_args,
)

# Egitimi baslat
print("[*] Egitim baslatiliyor...")
trainer.train()

# LoRA adaptoru kaydediliyor
trainer.model.save_pretrained("./triagecore-lora-adapter")
tokenizer.save_pretrained("./triagecore-lora-adapter")
# Faz 3 icin zip paketlemesi yapiliyor
!zip -r triagecore-lora-adapter.zip ./triagecore-lora-adapter
print("[+] Egitim tamamlandi ve LoRA adaptoru paketlendi!")